In [102]:
from sklearn.datasets import load_diabetes

import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [103]:
x , y = load_diabetes(return_X_y=True)

In [104]:
x_train ,x_test, y_train ,y_test = train_test_split(x , y ,  test_size=0.2 , random_state=2)

In [105]:
reg = LinearRegression()

reg.fit(x_train,y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [106]:
y_pred = reg.predict(x_test)
r2_score(y_test , y_pred)

0.439933866156897

In [107]:
print(reg.intercept_)
print(reg.coef_)

151.88331005254167
[  -9.15865318 -205.45432163  516.69374454  340.61999905 -895.5520019
  561.22067904  153.89310954  126.73139688  861.12700152   52.42112238]


In [108]:
x_train.shape[1]

10

# **Creating Own Stochastic GD**

In [109]:
class STOCH_GD :

    def __init__(self , learning_rate = 0.01 , epochs = 100):
        self.coef_ = np.ones(x_train.shape[1])
        self.intercept_ = 0 
        self.learning_rate = learning_rate
        self.epochs = epochs

    def fit(self , x_train , y_train):

        for i in range(self.epochs):
            for j in range(x_train.shape[0]):

                idx = np.random.randint(0, x_train.shape[0]) # We got diffrent result on same data because we select random value as idx for row 
                # BUt randomness we got is good enough if we are working on a big data and reach near to the solution 

                # Changeing intercepts 
                y_hat = np.dot(x_train[idx] ,self.coef_ ) + self.intercept_
                intercept_der =  -2 * (y_train[idx] - y_hat)
                self.intercept_ = self.intercept_ - (self.learning_rate * intercept_der)

                # Changing Coeff

                coef_der = -2 * np.dot((y_train[idx] - y_hat) ,x_train[idx])
                self.coef_ = self.coef_ - (self.learning_rate * coef_der)


        

    def predict(self , x_test):
        return np.dot(x_test,self.coef_) + self.intercept_

In [122]:
import time
start = time.time()
ST = STOCH_GD(epochs=10 , learning_rate=0.1)

print("Time taken " , time.time() - start)

Time taken  0.0


In [111]:
ST.fit(x_train ,y_train)

In [112]:
y_pred = ST.predict(x_test)
r2_score(y_test , y_pred)

0.4068731129622224

In [113]:
ST.coef_

array([  47.40804654, -168.41061821,  436.97727016,  322.10186624,
         -9.34988617,  -83.73838014, -193.73643534,  122.62465381,
        420.91261433,  128.0865395 ])

In [114]:
ST.intercept_

np.float64(167.9326674832115)

<div align="center">

<table>
<tr>
<td><p>b = 120 , m = -100</p>
<img src="animation4.gif" width="400"></td>
<td>x-axis = epochs , y-axis = loss , epoch =1 , but it going to 100 because there is 100 numbers <br> and our slop and intercept update 100 time <br>
In Stochastic  may be every next number is worst then  beyound because we re using random rows . Inconsistency  </p>
<img src="stochastic_animation_line_plot.gif" width="400"></td>
</tr>
<tr>
<td><p>
x-axis = m , y-axis = b , darker shade means minimum loss</p>
<img src="animation8.gif" width="400"></td>
<td><img src="stochastic_animation_contour_plot.gif" width="400"></td>
</tr>
</table>

</div>

# Comparing our STOCH_REg with SGD Regressor 

In [123]:
from sklearn.linear_model import SGDRegressor

In [124]:
SGD = SGDRegressor(learning_rate='constant' , max_iter=100 , eta0=0.01)

In [125]:
SGD.fit(x_train , y_train)

,"loss loss: str, default='squared_error'The loss function to be used. The possible values are 'squared_error','huber', 'epsilon_insensitive', or 'squared_epsilon_insensitive'The 'squared_error' refers to the ordinary least squares fit.'huber' modifies 'squared_error' to focus less on getting outlierscorrect by switching from squared to linear loss past a distance ofepsilon. 'epsilon_insensitive' ignores errors less than epsilon and islinear past that; this is the loss function used in SVR.'squared_epsilon_insensitive' is the same but becomes squared loss pasta tolerance of epsilon.More details about the losses formulas can be found in the:ref:`User Guide `.",'squared_error'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'l2'
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",0.0001
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with 0 <= l1_ratio <= 1.l1_ratio=0 corresponds to L2 penalty, l1_ratio=1 to L1.Only used if `penalty` is 'elasticnet'.Values must be in the range `[0.0, 1.0]` or can be `None` if`penalty` is not `elasticnet`... versionchanged:: 1.7 `l1_ratio` can be `None` when `penalty` is not ""elasticnet"".",0.15
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True
,"max_iter max_iter: int, default=1000The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the ``fit`` method, and not the:meth:`partial_fit` method.Values must be in the range `[1, inf)`... versionadded:: 0.19",100
,"tol tol: float or None, default=1e-3The stopping criterion. If it is not None, training will stopwhen (loss > best_loss - tol) for ``n_iter_no_change`` consecutiveepochs.Convergence is checked against the training loss or thevalidation loss depending on the `early_stopping` parameter.Values must be in the range `[0.0, inf)`... versionadded:: 0.19",0.001
,"shuffle shuffle: bool, default=TrueWhether or not the training data should be shuffled after each epoch.",True
,"verbose verbose: int, default=0The verbosity level.Values must be in the range `[0, inf)`.",0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-insensitive loss functions; only if `loss` is'huber', 'epsilon_insensitive', or 'squared_epsilon_insensitive'.For 'huber', determines the threshold at which it becomes lessimportant to get the prediction exactly right.For epsilon-insensitive, any differences between the current predictionand the correct label are ignored if they are less than this threshold.Values must be in the range `[0.0, inf)`.",0.1
,"random_state random_state: int, RandomState instance, default=NoneUsed for shuffling the data, when ``shuffle`` is set to ``True``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None


In [127]:
SGD_PRED = SGD.predict(x_test)

In [128]:
r2_score(y_test , SGD_PRED)

0.4238984513749344